In [ ]:
import pandas as pd
import numpy as np
import time
from datetime import datetime, date, time
import re
from IPython.display import clear_output
import calendar

In [ ]:
from clickhouse_driver import Client
client = Client(
    host='192.168.1.131',
    port=9000,
    user='user',
    password='pass',
    database='FINACLE'
)

In [ ]:
ResultPath = input('Үр дүнг хадгалах фолдерын зам: ')
date_Inp1 = input('Хамрах эхний өдөр(YYYY-MM-DD):')
date_Inp2 = input('Хамрах эцсийн өдөр (YYYY-MM-DD):')
try:
    start_Date = datetime.strptime(date_Inp1, "%Y-%m-%d").strftime("%d-%b-%Y")
    end_Date = datetime.strptime(date_Inp2, "%Y-%m-%d").strftime("%d-%b-%Y")
    print("Хамрах эхний өдөр =", start_Date)
    print("Хамрах эцсийн өдөр =", end_Date)
except ValueError:
    print("Date format error")

In [ ]:
BRANCH_SOL_ID='401'
branch_sol_id=401  #int

In [ ]:
query = f""" SELECT
            T3.H_TRAN_ID,
            T3.H_TRAN_DATE,
            T3.H_DTH_INIT_SOL_ID,
            T3.H_ENTRY_DATE,
            T3.H_ENTRY_USER_ID,
            T3.H_PSTD_DATE,
            T3.H_PSTD_USER_ID,
            T3.H_VFD_DATE,
            T3.H_VFD_USER_ID,
            T3.H_TRAN_TYPE,
            T3.H_TRAN_SUB_TYPE,
            T3.H_PART_TRAN_TYPE,
            T3.H_TRAN_AMT,
            T3.H_TRAN_CRNCY_CODE,
            ifNull(toString(T3.H_TRAN_PARTICULAR), '') AS H_TRAN_PARTICULAR,
            ifNull(toString(T3.B_CHANNEL_ID), '') AS B_CHANNEL_ID,
            ifNull(toString(T3.B_BANK), '') AS B_BANK,
            ifNull(toString(T3.B_TYPE), '') AS B_TYPE,
            ifNull(toString(T3.H_TRAN_RMKS), '') AS H_TRAN_RMKS,
            trim(T3.A_TRAN_ID) AS A_TRAN_ID,
            T3.H_ACID,
            T3.H_SOL_ID,
            T3.H_GL_SUB_HEAD_CODE,
            T3.B_ACCT_RATE * T3.H_TRAN_AMT AS H_TRAN_AMT_MNT,
            GAM.SCHM_CODE AS SCHM_CODE,
            T3.B_ACCT_PRTY_NUMBER,
            GAM.CIF_ID AS CIF_ID,
            GAM.ACCT_NAME AS ACCT_NAME
        FROM FINACLE.HTD_ATD T3
        LEFT JOIN (
            SELECT ACID, SCHM_CODE, CIF_ID, ACCT_NAME
            FROM FINACLE.GAM_ACCOUNTS
        ) AS GAM ON GAM.ACID = T3.H_ACID
        WHERE T3.H_TRAN_DATE between toDate('{date_Inp1}') AND toDate('{date_Inp2}')
        and  T3.H_DTH_INIT_SOL_ID ='{BRANCH_SOL_ID}'
"""
data, columns = client.execute(query, with_column_types=True)
ST7020 = pd.DataFrame(data, columns=[col[0] for col in columns]).drop_duplicates()

query_curr = f"""SELECT
            T3.H_TRAN_ID,
            T3.H_TRAN_DATE,
            T3.H_DTH_INIT_SOL_ID,
            T3.H_ENTRY_DATE,
            T3.H_ENTRY_USER_ID,
            T3.H_PSTD_DATE,
            T3.H_PSTD_USER_ID,
            T3.H_VFD_DATE,
            T3.H_VFD_USER_ID,
            T3.H_TRAN_TYPE,
            T3.H_TRAN_SUB_TYPE,
            T3.H_PART_TRAN_TYPE,
            T3.H_TRAN_AMT,
            T3.H_TRAN_CRNCY_CODE,
            ifNull(toString(T3.H_TRAN_PARTICULAR), '') AS H_TRAN_PARTICULAR,
            ifNull(toString(T3.B_CHANNEL_ID), '') AS B_CHANNEL_ID,
            ifNull(toString(T3.B_BANK), '') AS B_BANK,
            ifNull(toString(T3.B_TYPE), '') AS B_TYPE,
            ifNull(toString(T3.H_TRAN_RMKS), '') AS H_TRAN_RMKS,
            trim(T3.A_TRAN_ID) AS A_TRAN_ID,
            T3.H_ACID,
            T3.H_SOL_ID,
            T3.H_GL_SUB_HEAD_CODE,
            T3.B_ACCT_RATE * T3.H_TRAN_AMT AS H_TRAN_AMT_MNT,
            GAM.SCHM_CODE AS SCHM_CODE,
            T3.B_ACCT_PRTY_NUMBER,
            GAM.CIF_ID AS CIF_ID,
            GAM.ACCT_NAME AS ACCT_NAME
        FROM FINACLE.HTD_ATD_CURRENT T3
        LEFT JOIN (
            SELECT ACID, SCHM_CODE, CIF_ID, ACCT_NAME
            FROM FINACLE.GAM_ACCOUNTS
        ) AS GAM ON GAM.ACID = T3.H_ACID
        WHERE T3.H_TRAN_DATE between toDate('{date_Inp1}') AND toDate('{date_Inp2}')
        and  T3.H_DTH_INIT_SOL_ID ='{BRANCH_SOL_ID}'
"""
data2, columns = client.execute(query_curr, with_column_types=True)
ST7020_cur = pd.DataFrame(data2, columns=[col[0] for col in columns]).drop_duplicates()
ST7020=pd.concat([ST7020,ST7020_cur]).drop_duplicates()


In [ ]:
ST7020['GL_PREFIX'] = ST7020['H_GL_SUB_HEAD_CODE'].astype(str).str[:2]

BASE = ST7020[
    (ST7020['GL_PREFIX'].isin(['21', '22'])) &
    (~ST7020['H_TRAN_PARTICULAR']
        .fillna('')
        .str.upper()
        .str.contains("ШИМТГЭЛ")
     ) &
    (~ST7020['H_PSTD_USER_ID'].isna()) &
    (ST7020['A_TRAN_ID'].isna()) &
    (pd.to_datetime(ST7020['H_ENTRY_DATE']) >= start_Date) &
    (pd.to_datetime(ST7020['H_ENTRY_DATE']) <= end_Date)
].copy()

BASE['TRAN_DATE'] = pd.to_datetime(BASE['H_ENTRY_DATE']).dt.date


In [ ]:
D_SUM = (
    BASE[BASE['H_PART_TRAN_TYPE'] == 'D']
    .groupby(['TRAN_DATE', 'CIF_ID'])['H_TRAN_AMT_MNT']
    .sum()
    .reset_index(name='DAY_TOTAL_MNT')
)

D_FLAG = D_SUM[D_SUM['DAY_TOTAL_MNT'] >= 10_000_000]

S12_D = BASE.merge(
    D_FLAG[['TRAN_DATE', 'CIF_ID']],
    on=['TRAN_DATE', 'CIF_ID'],
    how='inner'
)

S12_D['FLAG_TYPE'] = 'D >= 10,000,000'


In [ ]:
C_SUM = (
    BASE[BASE['H_PART_TRAN_TYPE'] == 'C']
    .groupby(['TRAN_DATE', 'CIF_ID'])['H_TRAN_AMT_MNT']
    .sum()
    .reset_index(name='DAY_TOTAL_MNT')
)

C_FLAG = C_SUM[C_SUM['DAY_TOTAL_MNT'] >= 10_000_000]

S12_C = BASE.merge(
    C_FLAG[['TRAN_DATE', 'CIF_ID']],
    on=['TRAN_DATE', 'CIF_ID'],
    how='inner'
)

S12_C['FLAG_TYPE'] = 'C >= 10,000,000'


In [ ]:
S12_D = S12_D.drop(columns=['GL_PREFIX', 'FLAG_TYPE'], errors='ignore')
S12_C = S12_C.drop(columns=['GL_PREFIX', 'FLAG_TYPE'], errors='ignore')
S12_ = pd.concat([S12_D, S12_C], ignore_index=True)
SUMMARY = pd.concat([
    D_FLAG.assign(TYPE='D'),
    C_FLAG.assign(TYPE='C')
]).sort_values(['TRAN_DATE', 'CIF_ID', 'TYPE'])




In [ ]:

with pd.ExcelWriter(""+ResultPath+"/S12_"+BRANCH_SOL_ID+"_"+start_Date+"_"+end_Date+".xlsx"+"") as writer:
    S12_.to_excel(writer, sheet_name='S12_', index=False)


In [ ]:
query1 = f"""
SELECT
  eh.EMPLOYEE_CODE as domain,
  eh.SOL_ID as sol_id,
  eh.DEPARTMENT_NAME as department_name,
  eh.POSITION_NAME as position_name,
  eh.B_TXNDATE as b_txndate,
  fa.ORGKEY as cif,
  fa.NAME as name
FROM
  ERP.ERP_EMPLOYEE ee
  INNER JOIN ERP.EMPLOYEE_HISTORY eh ON eh.EMPLOYEE_CODE = ee.EMPLOYEE_CODE
  LEFT JOIN FINACLE.ACCOUNTS fa ON ee.STATE_REG_NUMBER = fa.STRFIELD12
WHERE
  ee.IS_ACTIVE = 1
  AND (
    eh.CURRENT_STATUS_ID = 1
    OR eh.CURRENT_STATUS_ID = 3
  )
  AND eh.B_TXNDATE = toDate('{date_Inp2}')
  and eh.SOL_ID ='{BRANCH_SOL_ID}'

"""
data, columns = client.execute(query1, with_column_types=True)
EMP_CIF = pd.DataFrame(data, columns=[col[0] for col in columns]).drop_duplicates()
EMP_CIF.columns=map(str.upper, EMP_CIF)

In [ ]:
query1 = f"""SELECT
  T.B_TXNDATE AS B_TXNDATE_H,
  T.G_SOL_ID AS G_SOL_ID_H,
  T.G_CIF_ID AS G_CIF_ID_H,
  T.B_SEGCODE AS B_SEGCODE_H,
  T.G_FORACID AS G_FORACID_H,
  T.G_ACCT_NAME AS G_ACCT_NAME_H,
  T.G_SCHM_CODE AS G_SCHM_CODE_H,
  T.G_ACCT_MGR_USER_ID AS G_ACCT_MGR_USER_ID_H,
  T.G_ACCT_CRNCY_CODE AS G_ACCT_CRNCY_CODE_H,
  CASE
      WHEN T.B_MAIN_CLASSIFICATION < T.B_SUB_CLASSIFICATION
           AND trimBoth(T.B_ASSET_CONTROL_FLG) = 'U'
      THEN replaceRegexpOne(toString(T.B_SUB_CLASSIFICATION), '^0+', '')
      ELSE replaceRegexpOne(toString(T.B_MAIN_CLASSIFICATION), '^0+', '')
  END AS CLASSIFICATION_H,
  T.B_RATE AS B_RATE_H,
  T.B_TXNBAL AS B_TXNBAL_H,
  T.B_TXNBAL * T.B_RATE AS B_TXNBAL_MNT_H,
  if(T.L_CHRGE_OFF_FLG = 'Y', 'Y', 'N') AS L_CHRGE_OFF_FLG_H,
  T.B_ADVDATE AS B_ADVDATE_H,
  T.L_EI_PERD_END_DATE AS L_EI_PERD_END_DATE_H,
  toInt32(dateDiff('month', T.B_ADVDATE, T.L_EI_PERD_END_DATE)) AS MONTHS_LOAN_H,
  T.L_DIS_AMT AS L_DIS_AMT_H,
  T.L_DIS_AMT * T.B_RATE AS L_DIS_AMT_MNT_H,
  T.L_INT_DMD_OS AS L_INT_DMD_OS_H,
  T.L_INT_DMD_OS * T.B_RATE AS L_INT_DMD_OS_MNT_H,
  (T.B_ACCRBINT + T.B_ACCRCINT + T.B_ACCRFINT) AS B_ACCINT_H,
  (T.B_ACCRBINT + T.B_ACCRCINT + T.B_ACCRFINT) * T.B_RATE AS B_ACCINT_MNT_H,
  T.L_SEC_STATUS_FLG AS L_SEC_STATUS_FLG_H,
  abs(T.B_FULL_RATE) AS B_FULL_RATE_H
FROM FINACLE.GAM_LAM AS T
LEFT JOIN FINACLE.GAC AS G ON G.G_ACID = T.G_ACID
WHERE
  T.G_ACCT_CLS_FLG = 'Y' and  T.B_TXNDATE between toDate('{date_Inp1}') AND toDate('{date_Inp2}')
        and  T.G_SOL_ID='{BRANCH_SOL_ID}'
"""
data, columns = client.execute(query1, with_column_types=True)
LNL2040 = pd.DataFrame(data, columns=[col[0] for col in columns]).drop_duplicates()

In [ ]:
#1 Богино хугацаанд ашиглагдсан зээл
LNL2040['B_TXNDATE_H'] = pd.to_datetime(LNL2040['B_TXNDATE_H'])
LNL2040['B_ADVDATE_H'] = pd.to_datetime(LNL2040['B_ADVDATE_H'])

S7_1=LNL2040[(LNL2040['G_SOL_ID_H']==BRANCH_SOL_ID)
             & (pd.to_datetime(LNL2040['B_TXNDATE_H']) >= (start_Date))
             & (pd.to_datetime(LNL2040['B_TXNDATE_H']) <= (end_Date))]
S7_1['DIFF_DATE']= (S7_1['B_TXNDATE_H']-S7_1['B_ADVDATE_H'])
S7_1 = S7_1[(S7_1['DIFF_DATE'].dt.days.between(0, 10)) |(S7_1['B_ADVDATE_H'].dt.day.isin([28, 29, 30, 31]))]
S7_1['LASTDAYS'] = S7_1['B_ADVDATE_H'].dt.day.apply(lambda x: "Тийм" if x in [28, 29, 30, 31] else "Үгүй")
S7_1_CIF_COUNT = LNL2040[(LNL2040['G_SOL_ID_H']==BRANCH_SOL_ID)
             & (pd.to_datetime(LNL2040['B_TXNDATE_H'].dt.date) >= (start_Date))
             & (pd.to_datetime(LNL2040['B_TXNDATE_H'].dt.date) <= (end_Date))].groupby('G_CIF_ID_H')['G_CIF_ID_H'].count().reset_index(name='CIF_COUNT')
S7_1=S7_1.merge(S7_1_CIF_COUNT, how='left', on='G_CIF_ID_H')
with pd.ExcelWriter(""+ResultPath+"/S1_"+BRANCH_SOL_ID+"_"+start_Date+"_"+end_Date+".xlsx"+"") as writer:
       S7_1.to_excel(writer, sheet_name="S7_1", index=False)
       S7_1_CIF_COUNT.to_excel(writer, sheet_name="S7_1_CIF_COUNT", index=False)

In [ ]:
query1 = f"""
SELECT
    G_CIF_ID               AS c1,
    G_FORACID              AS c2,
    G_SOL_ID               AS c3,
    G_ACCT_NAME            AS c4,
    G_SCHM_CODE            AS c5,
    ACCT_POA_AS_NAME       AS c6,
    ACCT_POA_AS_REC_TYPE   AS c7,
    NMA_KEY_ID             AS c8,
    CUST_RELTN_CODE        AS c9,
    START_DATE             AS c10
FROM FINACLE.GAM_TAM T336641
INNER JOIN FINACLE.AAS T286256
    ON T286256.ACID = T336641.G_ACID
WHERE
    T286256.ACCT_POA_AS_REC_TYPE != 'M'
    AND T286256.START_DATE >= toDateTime('{date_Inp1}')
    AND T286256.START_DATE <  toDateTime('{date_Inp2}')
ORDER BY
    G_SOL_ID,
    G_CIF_ID,
    G_ACCT_NAME,
    G_FORACID,
    G_SCHM_CODE,
    ACCT_POA_AS_REC_TYPE,
    NMA_KEY_ID,
    ALT1_ACCT_POA_AS_NAME,
    ACCT_POA_AS_NAME,
    CUST_RELTN_CODE,
    START_DATE

"""
data, columns = client.execute(query1, with_column_types=True)
ST4003 = pd.DataFrame(data, columns=[col[0] for col in columns]).drop_duplicates()
ST4003


In [ ]:

query1 = f"""
SELECT
    s_c11 AS c1,   -- Салбар
    s_c8  AS c2,   -- CIF ID
    s_c7  AS c3,   -- Дансны нэр
    s_c9  AS c4,   -- Данcны дугаар
    s_c10 AS c5,   -- Бүтээгдэхүүний код

    s_c4  AS c6,   -- ✅ Joint Name
    s_c5  AS c7,   -- Account Type
    s_c6  AS c8,   -- ✅ Joint CIF

    s_c3  AS c9,   -- Relation Code
    ''    AS c10,  -- Нэмэлт
    s_c1  AS c11   -- START_DATE
FROM
(

SELECT DISTINCT
    T286256.START_DATE            AS s_c1,
    T286256.CUST_RELTN_CODE       AS s_c3,
    T286256.ACCT_POA_AS_NAME      AS s_c4,   -- Joint Name
    T286256.ACCT_POA_AS_REC_TYPE  AS s_c5,   -- Account Type
    T286256.NMA_KEY_ID            AS s_c6,   -- ✅ Joint CIF
    T336641.G_ACCT_NAME           AS s_c7,
    T336641.G_CIF_ID              AS s_c8,
    CASE
        WHEN T336641.G_SCHM_CODE LIKE '%617%'
        THEN 'N/A'
        ELSE T336641.G_FORACID
    END                           AS s_c9,
    T336641.G_SCHM_CODE           AS s_c10,
    T336641.G_SOL_ID              AS s_c11

    FROM FINACLE.GAM_SMT T336641
    INNER JOIN FINACLE.AAS T286256
        ON T286256.ACID = T336641.G_ACID
    WHERE
        T286256.ACCT_POA_AS_REC_TYPE != 'M'
        AND T286256.START_DATE >= toDateTime('{date_Inp1}')
        AND T286256.START_DATE <  toDateTime('{date_Inp2}')
) sub
ORDER BY
    c1,c2,c3,c4,c5,c6,c7,c8,c9,c11


"""
data, columns = client.execute(query1, with_column_types=True)
ST4004 = pd.DataFrame(data, columns=[col[0] for col in columns]).drop_duplicates()
ST4004

In [ ]:
#2 Ажилтан өөрийн эрхээр өөрийн дансанд гүйлгээ хийх
# EMP cif8 ST4003, ST4004 Файлуудын байршлыг заах
S12_6=ST7020[(~ST7020['H_ENTRY_USER_ID'].str.contains("SR|SYSTEM|CDCI|EOD1|PAYSYS|ROBOTMOD1|^ROBOTVER",na=False))
             & (~ST7020['H_VFD_USER_ID'].str.contains("SR|SYSTEM|CDCI|EOD1|PAYSYS|ROBOTMOD1|^ROBOTVER", na=False))
             &(~ST7020['H_PSTD_USER_ID'].isna())
             & (ST7020['A_TRAN_ID'].isna())
             & (pd.to_datetime(ST7020['H_TRAN_DATE']) >= (start_Date))
             & (pd.to_datetime(ST7020['H_TRAN_DATE']) <= (end_Date))]

EMP_CIF['DOMAIN'] = EMP_CIF['DOMAIN'].astype(str)
ST4003=pd.read_csv(r"C:\Users\148015768\Documents\work\SALBAR-AUDIT-NEGDSEN\EHNII-8\data\402\ST4003_Хамтрантай дан (1).csv")
ST4004=pd.read_csv(r"C:\Users\148015768\Documents\work\SALBAR-AUDIT-NEGDSEN\EHNII-8\data\402\ST4004_Хамтрантай да (1).csv")
nametable1 = list(ST4003.columns)
nametable2 = list(ST4004.columns)
ST4003.columns = ["col1", "col2", "col3", "col4", "col5", "col6", "col7", "col8", "col9", "col10"]
ST4004.columns = ["col1", "col2", "col3", "col4", "col5", "col6", "col7", "col8", "col9", "col10", "col11"]
S12_6a = S12_6.merge(EMP_CIF, how='left', left_on='CIF_ID', right_on='CIF')
S12_6a = S12_6a[S12_6a['H_ENTRY_USER_ID'] ==S12_6a['DOMAIN']]

st4003 = ST4003[ST4003['col8'].isin(EMP_CIF[EMP_CIF['SOL_ID'] == BRANCH_SOL_ID]['CIF'])]
emp_rp_left = st4003[['col2', 'col8']].rename(columns={'col2': 'ACID', 'col8': 'RP_CIF'})

st4004 = ST4004[ST4004['col8'].isin(EMP_CIF[EMP_CIF['SOL_ID'] == BRANCH_SOL_ID]['CIF'])]
emp_rp_right = st4004[['col4', 'col8']].rename(columns={'col4': 'ACID', 'col8': 'RP_CIF'})

EMP_RP = pd.concat([emp_rp_left, emp_rp_right])

EMP_RP= EMP_RP.merge(EMP_CIF, how='inner', left_on='RP_CIF', right_on='CIF')
EMP_RP = EMP_RP[['DOMAIN', 'ACID']].drop_duplicates()
EMP_RP['ACID'] = EMP_RP['ACID'].astype(str)

S12_6b = S12_6.merge(EMP_RP, how='left', left_on='B_ACCT_PRTY_NUMBER', right_on='ACID')
S12_6b = S12_6b[S12_6b['H_ENTRY_USER_ID'] == S12_6b['DOMAIN']]

with pd.ExcelWriter(""+ResultPath+"/S2_"+BRANCH_SOL_ID+"_"+start_Date+"_"+end_Date+".xlsx"+"") as writer:
       S12_6a.to_excel(writer, sheet_name="EMP_OWN", index=False)
       S12_6b.to_excel(writer, sheet_name="EMP_RE", index=False)
del S12_6

In [ ]:
#3 Харилцагчийн гүйлгээг саатуулах-Харилцагчийн гүйлгээг 30-аас дээш минут баталгаажуулаагүй бүх датаг гаргана.
S11_1=ST7020[(~ST7020['H_PSTD_USER_ID'].isna())
             & (ST7020['A_TRAN_ID'].isna())
             & ((ST7020['B_CHANNEL_ID'] == "UFC") | ST7020['B_CHANNEL_ID'].isna())
             & (~ST7020['H_ENTRY_USER_ID'].str.contains("SR|SYSTEM|CDCI|EOD1|PAYSYS|ROBOTMOD1|^ROBOTVER", na=False))
             & (~ST7020['H_VFD_USER_ID'].str.contains("SR|SYSTEM|CDCI|EOD1|PAYSYS|ROBOTMOD1|^ROBOTVER", na=False))
             & (pd.to_datetime(ST7020['H_TRAN_DATE']) >= (start_Date))
             & (pd.to_datetime(ST7020['H_TRAN_DATE']) <= (end_Date))
             ]
S11_1['DIF_TIME'] = (S11_1['H_PSTD_DATE']) -(S11_1['H_ENTRY_DATE'])
S11_1['DIF_TIME'] = S11_1['DIF_TIME'].dt.total_seconds().astype(float)
S11_1=S11_1[S11_1['DIF_TIME'] >= 1800]

S11_1['DIF_TIME'] = pd.to_timedelta(S11_1['DIF_TIME'], unit='s')
S11_1['DIF_TIME']=S11_1['DIF_TIME'].apply(lambda x: f"{x.days*24+x.seconds // 3600:02d}:{x.seconds % 3600 // 60:02d}:{x.seconds % 60:02d}")

with pd.ExcelWriter(""+ResultPath+"/S3_"+BRANCH_SOL_ID+"_"+start_Date+"_"+end_Date+".xlsx"+"") as writer:
   S11_1.to_excel(writer, index=False)


In [ ]:
#3 Харилцагчийн гүйлгээг саатуулах-Харилцагчийн гүйлгээг 30-аас дээш минут баталгаажуулаагүй бүх датаг гаргана. SWIFT
# S11_1 = ST7020[
#     (~ST7020['H_PSTD_USER_ID'].isna())
#     & (ST7020['A_TRAN_ID'].isna())
#     & (
#         (ST7020['B_CHANNEL_ID'] == "UFC") |
#         (ST7020['B_TYPE'] == "SWIFT") |
#         (ST7020['B_CHANNEL_ID'].isna())
#       )
#     & (~ST7020['H_ENTRY_USER_ID'].str.contains("SR|SYSTEM|CDCI|EOD1|PAYSYS|ROBOTMOD1|^ROBOTVER", na=False))
#     & (~ST7020['H_VFD_USER_ID'].str.contains("SR|SYSTEM|CDCI|EOD1|PAYSYS|ROBOTMOD1|^ROBOTVER", na=False))
#     & (pd.to_datetime(ST7020['H_TRAN_DATE']) >= start_Date)
#     & (pd.to_datetime(ST7020['H_TRAN_DATE']) <= end_Date)
# ]

# S11_1['DIF_TIME'] = S11_1['H_PSTD_DATE'] - S11_1['H_ENTRY_DATE']
# S11_1['DIF_TIME'] = S11_1['DIF_TIME'].dt.total_seconds().astype(float)

# S11_1 = S11_1[S11_1['DIF_TIME'] >= 1800]

# S11_1['DIF_TIME'] = pd.to_timedelta(S11_1['DIF_TIME'], unit='s')
# S11_1['DIF_TIME'] = S11_1['DIF_TIME'].apply(
#     lambda x: f"{x.days*24 + x.seconds // 3600:02d}:{x.seconds % 3600 // 60:02d}:{x.seconds % 60:02d}"
# )

# with pd.ExcelWriter(f"{ResultPath}/S3_{BRANCH_SOL_ID}_{start_Date}_{end_Date}.xlsx") as writer:
#     S11_1.to_excel(writer, index=False)


In [ ]:
#4 Орлого, зарлагын хий гүйлгээ
S12_1=ST7020[(~ST7020['H_ENTRY_USER_ID'].str.contains("SR|SYSTEM|CDCI|EOD1|PAYSYS|ROBOTMOD1|^ROBOTVER",na=False))
             & (~ST7020['H_VFD_USER_ID'].str.contains("SR|SYSTEM|CDCI|EOD1|PAYSYS|ROBOTMOD1|^ROBOTVER", na=False))
             & (ST7020['H_GL_SUB_HEAD_CODE'].str.startswith("21") |ST7020['H_GL_SUB_HEAD_CODE'].str.startswith("22"))
             &(~ST7020['H_PSTD_USER_ID'].isna())
             & (ST7020['A_TRAN_ID'].isna())
             & (~ST7020['H_TRAN_PARTICULAR'].str.upper().str.contains("БЭЛЭН ГАРГАЛТЫН ШИМТГЭЛ|БЭЛЭН ГАРГАЛТЫН  ШИМТГЭЛ|БЭЛЭН ЗАРЛАГЫН ШИМТГЭЛ|КАРТЫН БЭЛЭН ГАРГАЛТЫН  ШИМТГЭЛ|БЭЛЭН БУС ШИЛЖҮҮЛГИЙН ШИМТГЭЛ|CHARGES FOR PORD CUSTOMER PAYMENT|ЖИЛИЙН ХУРААМЖ|ПИН КОД ЗАХ", na=False))
             & (pd.to_datetime(ST7020['H_TRAN_DATE']) >= (start_Date))
             & (pd.to_datetime(ST7020['H_TRAN_DATE']) <= (end_Date))
             ]
S12_1a=S12_1.groupby(['H_TRAN_DATE', 'H_ENTRY_USER_ID', 'B_ACCT_PRTY_NUMBER']).filter(lambda x: x['H_TRAN_AMT_MNT'].where(x['H_PART_TRAN_TYPE'] == 'C').sum(skipna=True) ==
                                                                                           x['H_TRAN_AMT_MNT'].where(x['H_PART_TRAN_TYPE'] == 'D').sum(skipna=True))

S12_1aa=S12_1a[S12_1a['H_PART_TRAN_TYPE']=='C']
grouped_data = S12_1aa.groupby(['H_TRAN_DATE', 'H_ENTRY_USER_ID', 'B_ACCT_PRTY_NUMBER'])['H_TRAN_AMT_MNT'].sum().reset_index()
grouped_data.rename(columns={'H_TRAN_AMT_MNT': 'SUMIF'}, inplace=True)
S12_1a = S12_1a.merge(grouped_data, how='left', on=['H_TRAN_DATE', 'H_ENTRY_USER_ID', 'B_ACCT_PRTY_NUMBER'])
S12_1a = S12_1a.sort_values(by=['B_ACCT_PRTY_NUMBER', 'H_ENTRY_DATE'])

S12_1b = (S12_1.groupby(['H_TRAN_DATE', 'B_ACCT_PRTY_NUMBER']).filter(lambda x: x['H_TRAN_AMT_MNT'].where(x['H_PART_TRAN_TYPE'] == 'C').sum(skipna=True) ==
                                                                           x['H_TRAN_AMT_MNT'].where(x['H_PART_TRAN_TYPE'] == 'D').sum(skipna=True)))
S12_1bb=S12_1b[S12_1b['H_PART_TRAN_TYPE']=='C']
grouped_data1 = S12_1bb.groupby(['H_TRAN_DATE', 'B_ACCT_PRTY_NUMBER'])['H_TRAN_AMT_MNT'].sum().reset_index()
grouped_data.rename(columns={'H_TRAN_AMT_MNT': 'SUMIF'}, inplace=True)
S12_1b = S12_1b.merge(grouped_data, how='left', on=['H_TRAN_DATE', 'B_ACCT_PRTY_NUMBER'])
S12_1b = S12_1b.sort_values(by=['B_ACCT_PRTY_NUMBER', 'H_ENTRY_DATE'])

S12_1b = S12_1b[~S12_1b[['H_TRAN_DATE', 'H_TRAN_ID']].isin(S12_1a[['H_TRAN_DATE', 'H_TRAN_ID']]).any(axis=1)]

with pd.ExcelWriter(""+ResultPath+"/S4_"+BRANCH_SOL_ID+"_"+start_Date+"_"+end_Date+".xlsx"+"") as writer:
   S12_1a.to_excel(writer, sheet_name="ONE_EMP_S12_1", index=False)
   S12_1b.to_excel(writer, sheet_name="ALL_EMP_S12_1", index=False)
del S12_1b, S12_1a

In [ ]:
import pandas as pd

# --- Шүүлт: системийн гүйлгээг хасах, хүний баталгаажуулалттай, шаардлагатай хоосон баганууд ---
S12_5 = ST7020[
    (~ST7020['H_ENTRY_USER_ID'].str.contains("SR|SYSTEM|CDCI|EOD1|PAYSYS|ROBOTMOD1|^ROBOTVER", na=False)) &
    (~ST7020['H_VFD_USER_ID'].str.contains("SR|SYSTEM|CDCI|EOD1|PAYSYS|ROBOTMOD1|^ROBOTVER", na=False)) &
    (~ST7020['H_PSTD_USER_ID'].isna()) &
    (ST7020['A_TRAN_ID'].isna()) &
    (ST7020['B_CHANNEL_ID']=='')  &

    (pd.to_datetime(ST7020['H_TRAN_DATE']) >= start_Date) &
    (pd.to_datetime(ST7020['H_TRAN_DATE']) <= end_Date)
]

# --- Хоосон H_ENTRY_DATE мөрүүдийг хасах ---
S12_5['TIME'] = pd.to_datetime(S12_5['H_ENTRY_DATE']).dt.strftime('%H:%M:%S')
S12_5 = S12_5[(S12_5['TIME'] < '09:00:00') | (S12_5['TIME'] > '18:00:00')]  #Салбар бүрийн ажлын цагаас хамаарч өөрчлөгдөнө.


# --- Excel руу бичих ---
output_file = f"{ResultPath}/S5_{BRANCH_SOL_ID}_{start_Date}_{end_Date}.xlsx"

if len(S12_5) > 1000000:
    chunksize = 500000
    writer = pd.ExcelWriter(output_file, engine='xlsxwriter')
    for i in range(0, len(S12_5), chunksize):
        S12_5.iloc[i:i+chunksize].to_excel(writer, sheet_name=f'Sheet{i//chunksize + 1}', index=False)
    writer.save()
else:
    with pd.ExcelWriter(""+ResultPath+"/S5_"+BRANCH_SOL_ID+"_"+start_Date+"_"+end_Date+".xlsx"+"") as writer:
       S12_5.to_excel(writer, index=False)


In [ ]:
#  #5 Ажлын бус цагаар гүйлгээ хийх- Гүйлгээ хийсэн цаг нь ажлын 9-18н цагийн хооронд биш датаг гаргасан
# S12_5=ST7020[(~ST7020['H_ENTRY_USER_ID'].str.contains("SR|SYSTEM|CDCI|EOD1|PAYSYS|ROBOTMOD1|^ROBOTVER",na=False)) #системийн гүйлгээнүүдийг хасна
#              & (~ST7020['H_VFD_USER_ID'].str.contains("SR|SYSTEM|CDCI|EOD1|PAYSYS|ROBOTMOD1|^ROBOTVER", na=False))
#              & (~ST7020['H_PSTD_USER_ID'].isna())
#              & (ST7020['A_TRAN_ID'].isna())
#              &(ST7020['B_CHANNEL_ID'].isna())  #UFC тодруулах
#              & (pd.to_datetime(ST7020['H_TRAN_DATE']) >= (start_Date))
#              & (pd.to_datetime(ST7020['H_TRAN_DATE']) <= (end_Date))
#              ]

# S12_5['TIME'] = pd.to_datetime(S12_5['H_ENTRY_DATE']).dt.strftime('%H:%M:%S')
# S12_5 = S12_5[(S12_5['TIME'] < '09:00:00') | (S12_5['TIME'] > '18:00:00')]  #Салбар бүрийн ажлын цагаас хамаарч өөрчлөгдөнө.

# if len(S12_5) > 1000000:
#     chunksize = 500000
#     writer = pd.ExcelWriter(""+ResultPath+"/S5_"+BRANCH_SOL_ID+"_"+start_Date+"_"+end_Date+".xlsx"+"")
#     for i, chunk in enumerate(pd.read_xlsx(S12_5, chunksize=chunksize)):
#       chunk.to_excel(writer, sheet_name=f'Sheet{i+1}', index=False)
#     writer.save()
# else:
#     with pd.ExcelWriter(""+ResultPath+"/S5_"+BRANCH_SOL_ID+"_"+start_Date+"_"+end_Date+".xlsx"+"") as writer:
#        S12_5.to_excel(writer, index=False)


In [ ]:
#6 АТМ-ын дансыг тохируулахгүй байх-
#Гүйлгээ хийгдэж буй дансны ЕД дугаар нь 18742 буюу АТМ-ын түр данснуудын хянасан гүйлгээнүүдийг авав.
S13_5=ST7020[(ST7020['H_GL_SUB_HEAD_CODE'].str.contains("18742"))
             &(~ST7020['H_PSTD_USER_ID'].isna())
             &(ST7020['A_TRAN_ID'].isna())
             & (ST7020['H_SOL_ID']==BRANCH_SOL_ID)
             & (pd.to_datetime(ST7020['H_TRAN_DATE']) >= (start_Date))
             & (pd.to_datetime(ST7020['H_TRAN_DATE']) <= (end_Date))
             ]
with pd.ExcelWriter(""+ResultPath+"/S6_"+BRANCH_SOL_ID+"_"+start_Date+"_"+end_Date+".xlsx"+"") as writer:
       S13_5.to_excel(writer, index=False)
del S13_5

In [ ]:
EMP_CIF=EMP_CIF[['DOMAIN','CIF','NAME','SOL_ID']]

In [ ]:
#8 Ажилтан харилцагчийн дансанд хамтрангаар бүртгүүлсэн эсэх
S12_8_ST4003 = ST4003[ST4003['col8'].isin(EMP_CIF[EMP_CIF['SOL_ID'] == BRANCH_SOL_ID]['CIF'])]
S12_8_ST4003=S12_8_ST4003.merge(EMP_CIF, how='left', left_on='col8', right_on='CIF')

nametable1=nametable1+['DOMAIN', 'CIF', 'NAME', 'SOL_ID']
S12_8_ST4003.columns=nametable1

S12_8_ST4004 = ST4004[ST4004['col8'].isin(EMP_CIF[EMP_CIF['SOL_ID'] == BRANCH_SOL_ID]['CIF'])]
S12_8_ST4004=S12_8_ST4004.merge(EMP_CIF, how='left', left_on='col8', right_on='CIF')
nametable2=nametable2+['DOMAIN', 'CIF', 'NAME', 'SOL_ID']
S12_8_ST4004.columns=nametable2
with pd.ExcelWriter(""+ResultPath+"/S8_"+BRANCH_SOL_ID+"_"+start_Date+"_"+end_Date+".xlsx"+"") as writer:
       S12_8_ST4003.to_excel(writer,sheet_name="S12_8_ST4003", index=False)
       S12_8_ST4004.to_excel(writer, sheet_name="S12_8_ST4004", index=False)

In [ ]:
# def get_pandas_df(data_name, parameters, connection_string):
#     dict = { "FNG2011" :
#                 """
#         SELECT T541416.SOL_ID,
#        SUBSTR (T541416.GL_SUB_HEAD_CODE, 1, 1)
#            AS GL_SUB_HEAD_CODE_FIRST,
#        SUBSTR (T541416.GL_SUB_HEAD_CODE, 1, 2)
#            AS GL_SUB_HEAD_CODE_SECOND,
#        SUBSTR (T541416.GL_SUB_HEAD_CODE, 1, 3)
#            AS GL_SUB_HEAD_CODE_THIRD,
#        T541416.GL_SUB_HEAD_CODE,
#        T541416.PLACEHOLDER,
#        TRIM (BOTH ' ' FROM T541416.GL_SUB_HEAD_CODE_NAME)
#            AS GL_SUB_HEAD_CODE_NAME,
#        T541416.SCHM_CODE,
#        T541416.B_SEGCODE,
#        T541416.MAIN_CLASSIFICATION,
#        T541416.SUB_CLASSIFICATION,
#        T541416.ACCT_CRNCY_CODE,
#        T541416.CUR_RATE,
#        T541416.BAL,
#        T541416.BAL_MNT,
#        T541416.DEPOSIT_PERIOD_MTHS,
#        T541416.TXNDATE
#   FROM FINACLE.TRAILBALANCE_NEW T541416            /* FACT_TRAILBALANCE_NEW */
#  WHERE (    T541416.TXNDATE = '"""+end_Date+"""'
#         AND T541416.SOL_ID ='"""+BRANCH_SOL_ID+"""'
#         AND T541416.BAL_MNT <> 0)
# UNION ALL
# SELECT T422874.SOL_ID,
#        SUBSTR (T422874.GL_SUB_HEAD_CODE, 1, 1)
#            AS GL_SUB_HEAD_CODE_FIRST,
#        SUBSTR (T422874.GL_SUB_HEAD_CODE, 1, 2)
#            AS GL_SUB_HEAD_CODE_SECOND,
#        SUBSTR (T422874.GL_SUB_HEAD_CODE, 1, 3)
#            AS GL_SUB_HEAD_CODE_THIRD,
#        T422874.GL_SUB_HEAD_CODE,
#        T422874.PLACEHOLDER,
#        TRIM (BOTH ' ' FROM T422874.GL_SUB_HEAD_CODE_NAME)
#            AS GL_SUB_HEAD_CODE_NAME,
#        T422874.SCHM_CODE,
#        T422874.B_SEGCODE,
#        T422874.MAIN_CLASSIFICATION,
#        T422874.SUB_CLASSIFICATION,
#        T422874.ACCT_CRNCY_CODE,
#        T422874.CUR_RATE,
#        T422874.BAL,
#        T422874.BAL_MNT,
#        T422874.DEPOSIT_PERIOD_MTHS,
#        T422874.TXNDATE
#   FROM FINACLE.TRAILBALANCE T422874                    /* FACT_TRAILBALANCE */
#  WHERE (    T422874.SOL_ID = '"""+BRANCH_SOL_ID+"""'
#         AND T422874.TXNDATE = '"""+end_Date+"""'
#         AND T422874.BAL <> 0)
# UNION ALL
# SELECT T554672.SOL_ID,
#        SUBSTR (T554672.GL_SUB_HEAD_CODE, 1, 1)    AS GL_SUB_HEAD_CODE_FIRST,
#        SUBSTR (T554672.GL_SUB_HEAD_CODE, 1, 2)    AS GL_SUB_HEAD_CODE_SECOND,
#        SUBSTR (T554672.GL_SUB_HEAD_CODE, 1, 3)    AS GL_SUB_HEAD_CODE_THIRD,
#        T554672.GL_SUB_HEAD_CODE,
#        T554672.PLACEHOLDER,
#        T554672.GL_SUB_HEAD_CODE_NAME              AS GL_SUB_HEAD_CODE_NAME,
#        T554672.SCHM_CODE,
#        T554672.B_SEGCODE,
#        T554672.MAIN_CLASSIFICATION,
#        T554672.SUB_CLASSIFICATION,
#        T554672.ACCT_CRNCY_CODE,
#        T554672.CUR_RATE,
#        T554672.BAL,
#        T554672.BAL_MNT,
#        T554672.DEPOSIT_PERIOD_MTHS,
#        T554672.TXNDATE
#   FROM FINACLE.SOL  T446798                                    /* DIM_SOL_1 */
#        INNER JOIN FINACLE.TRAILBALANCE_CORRECTION T554672 /* FACT_TRAILBALANCE_CORRECTION */
#            ON T446798.SOL_ID = T554672.SOL_ID
#  WHERE (    T554672.TXNDATE = '"""+end_Date+"""'
#         AND T554672.SOL_ID = '"""+BRANCH_SOL_ID+"""'
#         AND T554672.BAL <> 0)
#                                  """
#                          }
#     df = pd.read_sql_query(sql=dict.get(data_name, ""), params=parameters, con=connection_string)
#     return df
# FNG2011 = get_pandas_df(data_name="FNG2011", parameters={}, connection_string=engine)
# FNG2011.columns=map(str.upper, FNG2011)

In [ ]:
FNG2011=pd.read_csv(r"C:\Users\148015768\Documents\work\SALBAR-AUDIT-NEGDSEN\EHNII-8\data\402\FNG2011_Үлдэгдэл тэнцэл _ (1).csv")
# FNG2011=pd.read_excel(r"C:\Users\148015768\Documents\work\CODE\data\363\FNG2011_Үлдэгдэл_тэнцэл_363.xlsx")

In [ ]:
LNL2010=pd.read_csv(r"C:\Users\148015768\Documents\work\SALBAR-AUDIT-NEGDSEN\EHNII-8\data\402\LNL2010_Зээлийн дансны д (9).csv")
# LNL2010=pd.read_excel(r"C:\Users\148015768\Documents\work\SALBAR-AUDIT-NEGDSEN\EHNII-8\data\401\LNL2010.xlsx")
CDC2011=pd.read_csv(r"C:\Users\148015768\Documents\work\SALBAR-AUDIT-NEGDSEN\EHNII-8\data\402\CDC2011_Кредит картын дэ (4).csv")

In [ ]:
LNL2010.columns = [f"DESC{i+1}" for i in range(len(LNL2010.columns))]
LNL2010['DESC13']=pd.to_numeric(LNL2010['DESC13'], errors='coerce')

In [ ]:
#7 ЗТУБАХС-ийн хүрэлцээг тооцох
S6_30=FNG2011[(FNG2011['Салбар']==BRANCH_SOL_ID)
             &(FNG2011['ЕД left 3']=='169')]
S6_3_grouped=S6_30.groupby('Валют')
S6_3=S6_3_grouped['Үлдэгдэл дүн /валют/'].sum().reset_index(name='TOTAL')

#CDC2011['FUND_RATE_NUMBER']=pd.to_numeric(CDC2011['Сан байгуулах хувь'].str.replace("%", "", regex=True), errors='coerce') / 100
CDC2011['FUND_RATE_NUMBER'] = CDC2011['Сан байгуулах хувь'] * 100
lnl=pd.DataFrame()
lnl['Валют']=LNL2010[LNL2010['DESC1']==branch_sol_id]['DESC7']
lnl['FUND']=LNL2010[LNL2010['DESC1']==branch_sol_id]['DESC12']
lnl['FUND']=lnl['FUND']*(-1)

cdc=pd.DataFrame()
cdc = CDC2011[(CDC2011['Салбарын дугаар']==branch_sol_id)
              & (CDC2011['Насжилт']!='CR')].copy()

# Сан байгуулах хувь баганаас % тэмдэг арилгаж, тоон төрөлд хөрвүүлэх
cdc['FUND_RATE_NUMBER'] = pd.to_numeric(cdc['Сан байгуулах хувь'].astype(str).str.replace('%','',regex=False), errors='coerce') / 100

# FUND баганыг тооцох
cdc['FUND'] = cdc['Үлдэгдэл /MNT/'] * cdc['FUND_RATE_NUMBER'] * (-1)
cdc['Валют'] = cdc['Валют']

# lnl-т нэгтгэх
lnl = pd.concat([lnl, cdc[['Валют','FUND']]])

# Групп хийж нийлбэр гаргах
result = lnl.groupby('Валют')['FUND'].sum().reset_index(name='FUND_TOTAL')

# S6_3-т merge хийх
S6_3 = S6_3.merge(result, how='outer', on='Валют')

In [ ]:
# #7 ЗТУБАХС-ийн хүрэлцээг тооцох
# S6_30=FNG2011[(FNG2011['SOL_ID']==BRANCH_SOL_ID)
#              &(FNG2011['GL_SUB_HEAD_CODE_THIRD']=='169')]
# S6_3_grouped=S6_30.groupby('ACCT_CRNCY_CODE')
# S6_3=S6_3_grouped['BAL'].sum().reset_index(name='TOTAL')

# #CDC2011['FUND_RATE_NUMBER']=pd.to_numeric(CDC2011['Сан байгуулах хувь'].str.replace("%", "", regex=True), errors='coerce') / 100
# CDC2011['FUND_RATE_NUMBER'] = CDC2011['Сан байгуулах хувь'] * 100
# lnl=pd.DataFrame()
# lnl['ACCT_CRNCY_CODE']=LNL2010[LNL2010['DESC1']==branch_sol_id]['DESC7']
# lnl['FUND']=LNL2010[LNL2010['DESC1']==branch_sol_id]['DESC12']
# lnl['FUND']=lnl['FUND']*(-1)

# cdc=pd.DataFrame()
# cdc = CDC2011[(CDC2011['Салбарын дугаар']==branch_sol_id)
#               & (CDC2011['Насжилт']!='CR')].copy()

# # Сан байгуулах хувь баганаас % тэмдэг арилгаж, тоон төрөлд хөрвүүлэх
# cdc['FUND_RATE_NUMBER'] = pd.to_numeric(cdc['Сан байгуулах хувь'].astype(str).str.replace('%','',regex=False), errors='coerce') / 100

# # FUND баганыг тооцох
# cdc['FUND'] = cdc['Үлдэгдэл /MNT/'] * cdc['FUND_RATE_NUMBER'] * (-1)
# cdc['ACCT_CRNCY_CODE'] = cdc['Валют']

# # lnl-т нэгтгэх
# lnl = pd.concat([lnl, cdc[['ACCT_CRNCY_CODE','FUND']]])

# # Групп хийж нийлбэр гаргах
# result = lnl.groupby('ACCT_CRNCY_CODE')['FUND'].sum().reset_index(name='FUND_TOTAL')

# # S6_3-т merge хийх
# S6_3 = S6_3.merge(result, how='outer', on='ACCT_CRNCY_CODE')

In [ ]:
#LAST
#7 ЗТУБАХС-ийн хүрэлцээг тооцох
S7_=FNG2011[(FNG2011['Салбар']==BRANCH_SOL_ID)
             &(FNG2011['ЕД left 3']=='169')]
S7_grouped=S6_30.groupby('Валют')
S7_=S7_grouped['Үлдэгдэл дүн /валют/'].sum().reset_index(name='TOTAL')

#CDC2011['FUND_RATE_NUMBER']=pd.to_numeric(CDC2011['Сан байгуулах хувь'].str.replace("%", "", regex=True), errors='coerce') / 100
CDC2011['FUND_RATE_NUMBER'] = CDC2011['Сан байгуулах хувь'] * 100
lnl=pd.DataFrame()
lnl['Валют']=LNL2010[LNL2010['DESC1']==branch_sol_id]['DESC7']
lnl['FUND']=LNL2010[LNL2010['DESC1']==branch_sol_id]['DESC16']
lnl['FUND']=lnl['FUND']*(-1)

cdc=pd.DataFrame()
cdc['Валют']=CDC2011[(CDC2011['Салбарын дугаар']==branch_sol_id)
                               &(~CDC2011['Насжилт']=='CR')]['Валют']
cdc['FUND']=CDC2011[(CDC2011['Салбарын дугаар']==branch_sol_id)
                    &(~CDC2011['Насжилт']=='CR')]['Үлдэгдэл /MNT/']*CDC2011['FUND_RATE_NUMBER']*(-1)

lnl=pd.concat([lnl,cdc])
lnl_group=lnl.groupby('Валют')
result=lnl_group['FUND'].sum().reset_index(name='FUND_TOTAL')
S7_=S7_.merge(result, how='outer', on='Валют')

with pd.ExcelWriter(""+ResultPath+"/S7_"+BRANCH_SOL_ID+"_"+start_Date+"_"+end_Date+".xlsx"+"") as writer:
       S7_.to_excel(writer, index=False)
del S7_

In [ ]:
# #UMNUH
# #7 ЗТУБАХС-ийн хүрэлцээг тооцох
# S6_30=FNG2011[(FNG2011['SOL_ID']==BRANCH_SOL_ID)
#              &(FNG2011['GL_SUB_HEAD_CODE_THIRD']=='169')]
# S6_3_grouped=S6_30.groupby('ACCT_CRNCY_CODE')
# S6_3=S6_3_grouped['BAL'].sum().reset_index(name='TOTAL')

# #CDC2011['FUND_RATE_NUMBER']=pd.to_numeric(CDC2011['Сан байгуулах хувь'].str.replace("%", "", regex=True), errors='coerce') / 100
# CDC2011['FUND_RATE_NUMBER'] = CDC2011['Сан байгуулах хувь'] * 100
# lnl=pd.DataFrame()
# lnl['ACCT_CRNCY_CODE']=LNL2010[LNL2010['DESC1']==branch_sol_id]['DESC7']
# lnl['FUND']=LNL2010[LNL2010['DESC1']==branch_sol_id]['DESC16']
# lnl['FUND']=lnl['FUND']*(-1)

# cdc=pd.DataFrame()
# cdc['ACCT_CRNCY_CODE']=CDC2011[(CDC2011['Салбарын дугаар']==branch_sol_id)
#                                &(~CDC2011['Насжилт']=='CR')]['Валют']
# cdc['FUND']=CDC2011[(CDC2011['Салбарын дугаар']==branch_sol_id)
#                     &(~CDC2011['Насжилт']=='CR')]['Үлдэгдэл /MNT/']*CDC2011['FUND_RATE_NUMBER']*(-1)

# lnl=pd.concat([lnl,cdc])
# lnl_group=lnl.groupby('ACCT_CRNCY_CODE')
# result=lnl_group['FUND'].sum().reset_index(name='FUND_TOTAL')
# S6_3=S6_3.merge(result, how='outer', on='ACCT_CRNCY_CODE')

# with pd.ExcelWriter(""+ResultPath+"/S6_3_"+BRANCH_SOL_ID+"_"+start_Date+"_"+end_Date+".xlsx"+"") as writer:
#        S6_3.to_excel(writer, index=False)
# del S6_3